# Cálculo de Perfil de Elevación del Horizonte

Este notebook calcula el horizonte real visible desde un punto central dado, utilizando APIs gratuitas de elevación (`open-elevation` y `open-meteo`). El punto de referencia usado como ejemplo es la Universidad de Antioquia.

In [30]:
import numpy as np
import urllib.request
import urllib.error
import json
import time
import plotly.graph_objects as go
from math import radians, degrees, sin, cos, asin, atan2

## 1. Definición de funciones auxiliares
Funciones para calcular nuevas coordenadas dada una distancia y azimuth, y para consultar el API de elevación.

In [31]:
def get_destination_point(lat, lon, distance_km, bearing_deg):
    R = 6371.0 # Radio de la Tierra en km
    lat_rad = radians(lat)
    lon_rad = radians(lon)
    bearing_rad = radians(bearing_deg)
    
    lat2 = asin(sin(lat_rad)*cos(distance_km/R) + cos(lat_rad)*sin(distance_km/R)*cos(bearing_rad))
    lon2 = lon_rad + atan2(sin(bearing_rad)*sin(distance_km/R)*cos(lat_rad), cos(distance_km/R)-sin(lat_rad)*sin(lat2))
    
    return degrees(lat2), degrees(lon2)

def fetch_elevations(coords, source="open-elevation", chunk_size=100):
    elevations = []
    
    if source == "open-elevation":
        # Open-Elevation es más lento y requiere chunks más pequeños para evitar Timeouts
        chunk_size = min(chunk_size, 50)
        for i in range(0, len(coords), chunk_size):
            chunk = coords[i:i+chunk_size]
            locs_str = "|".join([f"{lat:.6f},{lon:.6f}" for lat, lon in chunk])
            url = f"https://api.open-elevation.com/api/v1/lookup?locations={locs_str}"
            
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "MontuPython/1.0"})
                with urllib.request.urlopen(req, timeout=30.0) as resp:
                    data = json.load(resp)
                    results = data.get("results", [])
                    for res in results:
                        elevations.append(float(res["elevation"]))
            except Exception as e:
                print(f"Error en chunk {i} (Open-Elevation): {e}")
                elevations.extend([float('nan')] * len(chunk))
            
            time.sleep(1.0)
            
    elif source == "open-meteo":
        # Open-Meteo tiene un límite estricto por minuto para peticiones de lote
        for i in range(0, len(coords), chunk_size):
            chunk = coords[i:i+chunk_size]
            lats_str = ",".join([f"{lat:.6f}" for lat, lon in chunk])
            lons_str = ",".join([f"{lon:.6f}" for lat, lon in chunk])
            url = f"https://api.open-meteo.com/v1/elevation?latitude={lats_str}&longitude={lons_str}"
            
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "MontuPython/1.0"})
                with urllib.request.urlopen(req, timeout=30.0) as resp:
                    data = json.load(resp)
                    elev_list = data.get("elevation", [])
                    elevations.extend(elev_list)
            except urllib.error.HTTPError as e:
                print(f"Error en chunk {i} (Open-Meteo): {e.code} {e.reason}")
                if e.code == 429:
                    try:
                        msg = json.loads(e.read().decode('utf-8'))
                        print("Detalle:", msg.get('reason', 'Límite de peticiones excedido'))
                    except:
                        pass
                elevations.extend([float('nan')] * len(chunk))
            except Exception as e:
                print(f"Error en chunk {i} (Open-Meteo): {e}")
                elevations.extend([float('nan')] * len(chunk))
                
            # Un retraso grande (10s) para evitar el límite por minuto de Open-Meteo (alrededor de 6 peticiones por min)
            time.sleep(10.0)
            
    return elevations

## 2. Configuración inicial y generación de malla cilíndrica
Tomamos como punto central la Universidad de Antioquia.

In [32]:
# Coordenadas y altura base (Universidad de Antioquia)
lat0 = 6.266152
lon0 = -75.569335
alt0 = 1468.0

# Parámetros de la malla
azimuths = np.arange(0, 360, 5) # Cada 5 grados
distances = np.arange(1, 31, 2) # De 1km a 30km, saltos de 2km

grid_coords = []
azimuth_dist_map = []

for az in azimuths:
    for d in distances:
        lat_d, lon_d = get_destination_point(lat0, lon0, d, az)
        grid_coords.append((lat_d, lon_d))
        azimuth_dist_map.append((az, d))

print(f"Total de puntos a consultar: {len(grid_coords)}")

Total de puntos a consultar: 1080


## 3. Consulta de elevaciones y cálculo del horizonte aparente
Aquí puedes elegir si usar `source="open-elevation"` o `source="open-meteo"`.

In [34]:
print("Consultando elevaciones...")

# Puedes cambiar el source a "open-meteo" si lo deseas, teniendo en cuenta sus límites por minuto
elevations = fetch_elevations(
    grid_coords, 
    source="open-elevation", 
    #source="open-meteo", 
    chunk_size=50
)

Consultando elevaciones...


In [35]:
horizon_profile = {}
for i, (az, d) in enumerate(azimuth_dist_map):
    alt = elevations[i]
    if np.isnan(alt):
        continue
    
    # Radio de la Tierra en metros
    R_e = 6371000.0
    
    # Distancias al centro de la Tierra
    r1 = R_e + alt0
    r2 = R_e + alt
    
    # Ángulo central entre los dos puntos
    distance_m = d * 1000.0
    theta = distance_m / R_e
    
    # Coordenadas locales respecto al observador (y=arriba, x=adelante)
    y = r2 * np.cos(theta) - r1
    x = r2 * np.sin(theta)
    
    # Ángulo de elevación real teniendo en cuenta la curvatura
    elev_angle = degrees(np.arctan2(y, x))
    
    if az not in horizon_profile:
        horizon_profile[az] = elev_angle
    else:
        if elev_angle > horizon_profile[az]:
            horizon_profile[az] = elev_angle

## 4. Visualización con Plotly
Construimos el gráfico del panorama de elevaciones.

In [36]:
plot_azimuths = sorted(list(horizon_profile.keys()))
plot_elevations = [horizon_profile[az] for az in plot_azimuths]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_azimuths, 
    y=plot_elevations, 
    mode='lines+markers', 
    name='Elevación del Horizonte',
    line=dict(shape='spline', smoothing=1.3)
))

fig.update_layout(
    title='Perfil de Elevación del Horizonte - Universidad de Antioquia',
    xaxis_title='Azimuth (grados, 0=N, 90=E, 180=S, 270=W)',
    yaxis_title='Ángulo de Elevación (grados)',
    xaxis=dict(tickmode='linear', tick0=0, dtick=45, range=[0, 360]),
    template='plotly_dark'
)
fig.show()